MLflow setup:
* Tracking server: GCP VM
* Backend store: GCP Cloud SQL postgresql database
* Artifacts store: GCP bucket

The experiments can be explored by accessing the remote server.

In [1]:
import mlflow
import os

# Jupyter lab server is currently running on the same VM as the MLFlow
# but MLFlow server endpoint is also served by Tailscale and available to all devices in the tailnet.

# TRACKING_SERVER_HOST = "mlops-vm.tailXXXXX.ts.net"

# In this notebook we will use VM's local net to access the MLFlow server.

TRACKING_SERVER_ENDPOINT = "https://mlops-vm.tailc0798c.ts.net:5000"
mlflow.set_tracking_uri(TRACKING_SERVER_ENDPOINT)

In [2]:
mlflow.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/5', creation_time=1786731794129, effective_trace_archival_retention=None, experiment_id='5', last_update_time=1786731794129, lifecycle_stage='active', name='nyc-taxi', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1786723449402, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1786723449402, lifecycle_stage='active', name='new-experiment', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='gs://$GCP_BUCKET/0', creation_time=1786645869124, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1786645869124, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("new-experiment")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2026/08/14 16:49:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


default artifacts URI: 'mlflow-artifacts:/2/f2d651e2d5c74729a96a1a8934bf462f/artifacts'
🏃 View run adaptable-donkey-753 at: https://mlops-vm.tailc0798c.ts.net:5000/#/experiments/2/runs/f2d651e2d5c74729a96a1a8934bf462f
🧪 View experiment at: https://mlops-vm.tailc0798c.ts.net:5000/#/experiments/2


In [4]:
from mlflow.tracking import MlflowClient


client = MlflowClient(TRACKING_SERVER_ENDPOINT)

In [6]:
run_id = client.search_runs(experiment_ids=['2'])[0].info.run_id
mlflow.register_model(
    model_uri=f"runs:/{run_id}/models",
    name='iris-classifier'
)

Registered model 'iris-classifier' already exists. Creating a new version of this model...
2026/08/14 16:50:38 WARNING mlflow.tracking._model_registry.fluent: Run with id f2d651e2d5c74729a96a1a8934bf462f has no artifacts at artifact path 'models', registering model based on models:/m-9faddff55d3648dbb2a48fd1f1e0c76c instead
2026/08/14 16:50:38 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classifier, version 2
Created version '2' of model 'iris-classifier'.


<ModelVersion: aliases=[], creation_timestamp=1786747838117, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1786747838117, metrics=None, model_id=None, name='iris-classifier', params=None, run_id='f2d651e2d5c74729a96a1a8934bf462f', run_link='', source='models:/m-9faddff55d3648dbb2a48fd1f1e0c76c', status='READY', status_message=None, tags={}, user_id='', version='2', workspace='default'>